# Songs 1970–1980: genres, lyrics and networks

MusicBrainz songs (metadata + genres) matched to Genius lyrics. This notebook shows the key results; the code lives in [src/](src/) (see [README.md](README.md)).

1. **Data overview**: what survives cleaning and matching, and do the two sources agree on genre?
2. **Genre network**: songs linked by shared genres, and its communities
3. **Lyrics network**: songs linked by similar lyrics. Do the communities match the genres?
4. **Genre classification**: can lyrics predict a song's genre?

Run top to bottom. The processed data is loaded from `data/processed/`, and it's rebuilt from `data/raw/` only if missing or when `REBUILD = True`. A full run takes roughly 10 minutes, mostly the two networks.

In [ ]:
REBUILD = False  # True: re-run cleaning + lyrics matching from data/raw/ (a few minutes)

import pandas as pd
from IPython.display import Markdown, display

from src import classification, exploration, genre_network, lyrics_network
from src.config import FIG_DIR, YEARS
from src.data import load_data

pd.set_option("display.max_colwidth", 120)
%matplotlib inline

songs, genius = load_data(rebuild=REBUILD)
print(f"MusicBrainz songs {YEARS}: {len(songs)}   ({songs.lyrics.notna().sum()} with lyrics)")
print(f"Genius songs {YEARS}: {len(genius)}")

## 1. Data overview

### How many songs survive each step?
**MusicBrainz:** raw recordings, merged into songs. We then count how many have a genre, how many of those genres map onto a Genius tag, how many got lyrics, and how many have labels from both sources that agree.

In [ ]:
exploration.musicbrainz_funnel(songs)

**Genius:** the cleaning steps, plus the subsets used in the network and classification analyses (rap is dropped there, since there are too few songs).

In [ ]:
exploration.genius_funnel(genius)

### How were the lyrics matched?
Exact title + artist first. Looser rules are only tried when that fails.

In [ ]:
songs.match.value_counts().to_frame("songs")

### Do MusicBrainz and Genius agree on the genre?
Rows: the Genius tag. Columns: the MusicBrainz genres mapped onto the same five tags (`mb_label`). Only songs that have both are counted. The diagonal is agreement.

In [ ]:
agree = exploration.label_agreement(songs)
display(agree)
print(f"Agreement: {songs.labels_agree.mean():.0%} of {songs.labels_agree.notna().sum()} songs with both labels")

### The dataset at a glance
Songs per year, the most common primary genres and release countries, and where lyrics were found. The bottom right is the agreement table above, as row percentages.

In [ ]:
exploration.plot_overview(songs, genius);

### The lyrics
Length and chorus repetition per Genius tag. *Distinctive words*: how many times more of the tag's songs use the word than songs of the other tags (counted per song). Rap is left out of that row, since it has too few matched songs.

In [ ]:
lyr = exploration.lyric_stats(songs)
exploration.plot_lyrics(lyr, exploration.distinctive_words(lyr));

## 2. Genre network

Each node is a song with at least one MusicBrainz genre. Two songs are linked when their genre lists overlap; edge weight is the number of shared genres. Each genre adds at most 500 randomly sampled links, so rock doesn't swamp the graph. Greedy modularity finds the communities.

To read the numbers: modularity runs from 0 (no community structure) to 1. *Modularity of primary genres* scores the primary genre labels used as communities, for comparison. *Purity* is the share of a community that has its most common primary genre.

In [ ]:
stats, summary, _, _ = genre_network.analyse(songs)
display(stats.round(3).to_frame("genre network"))
display(Markdown("**Communities with at least 20 songs**"))
display(summary.style.format({"purity": "{:.0%}"}))
display(Markdown(f"Interactive view (open in a browser; hover a song to see its links): "
                 f"[figures/genre_network.html]({(FIG_DIR / 'genre_network.html').relative_to(FIG_DIR.parent).as_posix()})"))

## 3. Lyrics similarity network

Each song is linked to the 10 songs with the most similar lyrics (TF-IDF cosine similarity). Songs by the same artist are never linked, so a community can't just be one artist's catalogue. Louvain finds communities **without seeing the genres**; we then check how well they line up with the genres. Sets larger than 8,000 songs are randomly sampled.

How to read the table:
- **genre Q**: modularity of the genres used as communities (0 = genres spread at random)
- **same-genre edges**: the share of links joining two songs of the same genre, against the share expected at random
- **NMI / ARI**: agreement between communities and genres (0 = none, 1 = identical)
- **purity**: compare it with the *purity of one big blob*, the score you get by putting every song in one community

Three song sets: all Genius songs, Genius without pop (pop is the largest and least distinct tag), and MusicBrainz songs where both sources agree on the genre.

In [ ]:
metrics, results = lyrics_network.run_all(songs, genius, verbose=False)
metrics.style.format("{:.3f}").format("{:.0f}", subset=pd.IndexSlice[["songs", "edges", "components", "communities"], :])

**Genre mix and typical words per community** (communities with at least 10 songs; each row sums to 100%)

In [ ]:
for name, r in results.items():
    display(Markdown(f"**{name}**"))
    display(r["summary"])

The network for Genius songs without pop, in the same layout twice: coloured by Louvain community (left) and by genre (right). If lyrics separated the genres, the two panels would look alike. An interactive version is written to `figures/lyrics_network.html`.

In [ ]:
r = results["Genius, without pop"]
lyrics_network.draw(r["G"], r["community"], r["genre"],
                    f"Lyrics similarity network, Genius {YEARS} without pop (each song linked to its {lyrics_network.K} most similar)");

## 4. Genre classification from lyrics

Three predictors, cross-validated with 5 folds. All of an artist's songs stay on the same side of each split, so a model can't score by recognising an artist.
- **guess**: random guessing in proportion to how common each genre is (the baseline)
- **counts**: hand-made features: length, vocabulary richness, repeated lines, word length
- **words**: TF-IDF word weights + logistic regression

Scores are macro-F1 (1.0 = perfect). Every genre counts equally, so always answering "pop" doesn't pay off.

In [ ]:
scores, confusions = classification.run_all(songs, genius, verbose=False)
scores.style.format({"guess": "{:.2f}", "counts": "{:.2f}", "words": "{:.2f}"})

**Where the word model goes wrong** on the Genius songs. Rows are the true genre and columns the prediction, as row percentages, so the diagonal is the recall per genre.

In [ ]:
for name, confusion in confusions.items():
    display(Markdown(f"**{name}**"))
    display(confusion.div(confusion.sum(axis=1), axis=0).style.format("{:.0%}").background_gradient(cmap="Blues", vmin=0, vmax=1))